# Day 4 — Experiment Tracking & Evaluation

---

Fine-tuning is **iterative**. You'll run 10, 20, 50 experiments before you're happy. Without tracking, you can't tell which hyperparameter combo won.

Today:

1. **Weights & Biases (W&B)** — the standard experiment-tracking tool
2. **Evaluation** — automatic (perplexity, exact match) + LLM-as-judge
3. Why BLEU / ROUGE are barely useful anymore


## 1. W&B in one paragraph

You install `wandb`, get a free API key, and add `report_to="wandb"` to your training config. Every hyperparameter, every loss value, every checkpoint gets logged to a beautiful web dashboard. You can compare runs side-by-side, group by config, and share the results with a URL.

For freshers: **use W&B for every fine-tuning experiment.** It costs nothing and the workflow benefits are enormous.


In [ ]:
!pip install wandb --quiet

In [ ]:
import os, wandb

# Set your key from https://wandb.ai/authorize
os.environ["WANDB_API_KEY"] = "..."  # replace or put in .env
wandb.login()


In [ ]:
# Enable W&B in your SFTConfig (Colab / GPU cell)
# from trl import SFTConfig
# config = SFTConfig(
#     output_dir="out",
#     report_to="wandb",                # <-- this line
#     run_name="triage-r16-lr2e-4",     # <-- searchable name
#     ...
# )


After training, open `https://wandb.ai/<you>/<project>` — you'll see:

- Loss curves (train + eval)
- All your hyperparameters as columns
- System stats (GPU mem, throughput)
- Every run in a table you can filter

**Naming convention worth adopting:** `<task>-r<rank>-lr<lr>-ep<epochs>`, e.g. `triage-r16-lr2e-4-ep3`. Future-you will thank present-you.


## 2. Evaluation — the honest kind

**The metric that actually matters: does the model do the right thing on inputs it hasn't seen?**

Three levels of eval, in the order you should use them:

### Level 1 — Manual spot-check
Grab 10-20 held-out examples. Run the model. Read each output. Score correct / partial / wrong.

**Do this before every "is my fine-tune better?" claim.** Nothing beats reading outputs with your own eyes.


In [ ]:
from transformers import pipeline

# Load the base and the fine-tuned adapter (Colab)
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct", ...)
# ft = PeftModel.from_pretrained(base, "triage-lora")

def spot_check(model, tokenizer, eval_examples, n=10):
    for row in eval_examples[:n]:
        msgs = row["messages"]
        expected = msgs[-1]["content"]                       # assistant turn
        prompt_msgs = msgs[:-1]                              # everything before

        inputs = tokenizer.apply_chat_template(
            prompt_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(inputs, max_new_tokens=20, do_sample=False)
        got = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

        mark = "OK " if got.lower() == expected.lower() else "!! "
        print(f"{mark} expected={expected!r:20} got={got!r}")

print("(paste the eval examples in a list and call spot_check)")


### Level 2 — Exact match / regex accuracy
For classification and structured-output tasks, you can auto-score.


In [ ]:
def accuracy(model, tokenizer, eval_examples):
    correct = 0
    for row in eval_examples:
        expected = row["messages"][-1]["content"].strip().lower()
        # ... generate got ...
        # if got.lower() == expected: correct += 1
        pass
    return correct / len(eval_examples)


### Level 3 — LLM-as-judge
When outputs are free-form (summaries, code, chat), have GPT-4 or Claude judge if the answer is *good enough*.

Prompt template:

```
You are a strict evaluator. Given a question, an expected answer, and a model answer,
return a JSON object {"score": 0-5, "reason": "..."}.

Question: <q>
Expected: <exp>
Model:    <got>
```

Use LLM-as-judge sparingly — it costs money per eval example — but it's the most flexible.


## 3. Why BLEU / ROUGE are barely used anymore

- **BLEU / ROUGE** measure n-gram overlap with a reference. They were the standard in the 2010s.
- Modern LLM outputs can be *correct but worded very differently* from the reference → BLEU says "bad" when it's fine.
- LLM-as-judge is now the standard for free-form eval.

**Know their names for interviews. Don't use them for RAG or general chat fine-tuning.** They're still fine for narrow tasks like translation.


## 4. A/B compare fine-tuned vs base

The one comparison that always matters:


In [ ]:
# Pseudocode - swap in your loaders
# base_acc = accuracy(base_model, tokenizer, eval_examples)
# ft_acc   = accuracy(ft_model,   tokenizer, eval_examples)
# print(f"base: {base_acc:.1%}   fine-tuned: {ft_acc:.1%}   delta: +{(ft_acc-base_acc)*100:.1f} pp")


If your fine-tune isn't beating the base model on your eval set, **do not ship it**. Either:

- More/better data
- Different hyperparameters (rank, LR, epochs)
- Or accept that the base model was already good enough (this is common!)


## Recap

- Track every fine-tune in **W&B**. It's free and makes comparing experiments trivial.
- Always eval on a **held-out set**. Never on train.
- Start with **manual spot-checks**, then move to **exact match** or **LLM-as-judge**.
- BLEU/ROUGE mostly obsolete for modern LLM work. Know the names, don't use them.
- **Ship only if fine-tune > base on your eval set.**
- **Next class:** merging, exporting, and serving your fine-tuned model.
